# Lab 16-03 · The DFT in Practice

Five parts, each built on `dft.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import dft

## Part 1 · The DFT matrix

In [2]:
N = 8
F = dft.dft_matrix(N)
print("F^H F == 8 I:", np.allclose(F.conj().T @ F, N * np.eye(N)))
x = np.random.default_rng(0).normal(size=N)
print("dft == numpy.fft.fft:", np.allclose(dft.dft(x), np.fft.fft(x)))
fig, axs = plt.subplots(2, 4, figsize=(10, 3.5), sharey=True)
for k, ax in enumerate(axs.flat):
    ax.stem(np.arange(N), F[k].real, basefmt=" ")
    ax.set_title(f"row {k}: cos(2 pi {k} n / 8)", fontsize=8)
plt.tight_layout(); plt.show()

F^H F == 8 I: True
dft == numpy.fft.fft: True


## Part 2 · The DFT samples the DTFT; padding interpolates

*Predict:* can zero padding separate two tones 0.5 bins apart?

In [3]:
pulse = np.hanning(18)[1:-1]                      # 16 samples
w = np.linspace(0, 2 * np.pi, 2000, endpoint=False)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(w, np.abs(dft.dtft(pulse, w)), color="0.6", label="DTFT")
for Np, mk in ((16, "o"), (64, "."), (256, ",")):
    X = np.fft.fft(pulse, Np)
    ax.plot(2 * np.pi * np.arange(Np) / Np, np.abs(X), mk, label=f"{Np}-point DFT")
ax.set_xlabel("w (rad/sample)"); ax.legend(fontsize=8); plt.show()

L = 64
n = np.arange(L)
x2 = np.cos(2 * np.pi * 10 * n / L) + np.cos(2 * np.pi * 10.5 * n / L)
X2 = np.abs(np.fft.fft(x2, 4096))[:2048]
f = np.arange(2048) * L / 4096                    # in units of 64-point bins
seg = X2[(f > 7) & (f < 14)]
peaks = np.sum((seg[1:-1] > seg[:-2]) & (seg[1:-1] > seg[2:]) & (seg[1:-1] > 0.5 * seg.max()))
print("major peaks between bins 7 and 14 after padding to 4096:", peaks)

major peaks between bins 7 and 14 after padding to 4096: 1


## Part 3 · Wrap-around

In [4]:
x3 = np.random.default_rng(1).normal(size=12)
h3 = np.random.default_rng(2).normal(size=5)
lin = np.convolve(x3, h3)
for Nc in range(12, 19):
    y = np.real(np.fft.ifft(np.fft.fft(x3, Nc) * np.fft.fft(h3, Nc)))
    ref = np.zeros(Nc); ref[:min(Nc, 16)] = lin[:min(Nc, 16)]
    err = np.max(np.abs(y[:min(Nc, 16)] - lin[:min(Nc, 16)])) if Nc < 16 else np.max(np.abs(y[:16] - lin))
    print(f"N = {Nc}: max error {err:.2e}")

N = 12: max error 1.28e+00
N = 13: max error 1.28e+00
N = 14: max error 1.28e+00
N = 15: max error 9.84e-01
N = 16: max error 8.88e-16
N = 17: max error 8.88e-16
N = 18: max error 1.78e-15


## Part 4 · Block filtering

In [5]:
x4 = np.random.default_rng(3).normal(size=100_000)
h4 = np.random.default_rng(4).normal(size=101)
t0 = time.perf_counter(); y_ref = np.convolve(x4, h4); t1 = time.perf_counter()
y_ola = dft.overlap_add(x4, h4, 1024); t2 = time.perf_counter()
y_ols = dft.overlap_save(x4, h4, 1024); t3 = time.perf_counter()
print(f"overlap-add max error {np.max(np.abs(y_ola - y_ref)):.1e}, overlap-save {np.max(np.abs(y_ols - y_ref)):.1e}")
print(f"times (s): numpy.convolve {t1 - t0:.3f}, OLA {t2 - t1:.3f}, OLS {t3 - t2:.3f}  (Python loop overheads dominate here)")
Ns = [2**p for p in range(7, 14)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogx(Ns, [dft.fft_conv_cost(101, Nn) for Nn in Ns], "o-", label="FFT block convolution")
ax.axhline(101, color="k", ls=":", label="direct (101)")
ax.set_xlabel("FFT size N"); ax.set_ylabel("real multiplies per output"); ax.legend(fontsize=8); plt.show()

overlap-add max error 2.1e-14, overlap-save 1.8e-14
times (s): numpy.convolve 0.002, OLA 0.008, OLS 0.004  (Python loop overheads dominate here)


## Part 5 · The cyclic prefix

*Predict:* at which prefix length does the error vector vanish, and do QPSK decisions fail before that?

In [6]:
def ofdm(Nsc, G, chan, nsym=200, seed=5):
    r = np.random.default_rng(seed)
    bits = r.integers(0, 2, (nsym, Nsc, 2))
    sym = ((2 * bits[..., 0] - 1) + 1j * (2 * bits[..., 1] - 1)) / math.sqrt(2)
    tx = []
    for s in sym:
        t = np.fft.ifft(s)
        tx.append(np.concatenate([t[Nsc - G:], t]) if G else t)
    rx = np.convolve(np.concatenate(tx), chan)
    C = np.fft.fft(chan, Nsc)
    e2, errs = 0.0, 0
    for i in range(nsym):
        est = np.fft.fft(rx[i * (Nsc + G) + G: i * (Nsc + G) + G + Nsc]) / C
        e2 += np.sum(np.abs(est - sym[i]) ** 2)
        errs += int(np.sum((np.sign(est.real) != np.sign(sym[i].real)) | (np.sign(est.imag) != np.sign(sym[i].imag))))
    return 100 * math.sqrt(e2 / (nsym * Nsc)), errs


chan = np.array([1.0, 0.5, -0.3, 0.2, 0.1])
for G in range(7):
    evm, errs = ofdm(64, G, chan)
    print(f"prefix {G}: rms error vector {evm:8.3g}%, wrong decisions {errs:4d}, overhead {100 * G / (64 + G):.2f}%")

prefix 0: rms error vector     33.9%, wrong decisions  233, overhead 0.00%
prefix 1: rms error vector     15.3%, wrong decisions   41, overhead 1.54%
prefix 2: rms error vector     5.87%, wrong decisions    0, overhead 3.03%
prefix 3: rms error vector     3.46%, wrong decisions    0, overhead 4.48%
prefix 4: rms error vector 4.22e-14%, wrong decisions    0, overhead 5.88%
prefix 5: rms error vector 4.22e-14%, wrong decisions    0, overhead 7.25%
prefix 6: rms error vector 4.22e-14%, wrong decisions    0, overhead 8.57%


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.